### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels- do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=documentation

Os dados extraídos:
<pre>
- Temperatura: variável 2m_temperature (t2m), retorna a temperatura em Kelvin
- Para a camada Bronze deverá ser mantida em Kelvin
- Para a camada Silver será necessário conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano, Mês e dia

Os dados requisitados estão no retangulo geográfico [6, -74, -34, -31] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil

In [ ]:
import sys, os
from datetime import datetime
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto 
sys.path.append(os.path.abspath(os.path.join('..')))

import copernicus_cds_utils as utils 

In [ ]:
# Cria uma conexão Spark 
spark = utils.get_spark_session("Temperatura")

In [ ]:
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

In [ ]:
def transform_data(df_temperatura):
    drop_cols = ["valid_time", "t2m", "number"]

    df_temperatura_final = \
        (df_temperatura
            .withColumns({"data_medicao"    : F.col("valid_time").cast("date")
                         ,"indicador"       : F.lit("temperatura") 
                         ,"valor"           : (F.col("t2m") - F.lit(273.15)).cast("double") # Converte a temperatura de Kelvin para Celsius
                         ,"unidade_medida"  : F.lit("celsius")})
            .drop(*drop_cols)
    )

    df_temperatura_final = \
        (df_temperatura_final
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

    return df_temperatura_final

In [ ]:
dataset_     = "reanalysis-era5-single-levels"
product_type = "reanalysis"
variable     = "2m_temperature"
ano          = 2026

years_process = range(1991,2027)

for ano in years_process:
    start = datetime(2026, 8, 16).now()
    
    print("Obter dados, converter e gravar para o ano : ",ano, " - ", start, end="" )

    # Recupera os dados do CDS em formato netcdf4 -> Landing
    ret_download = utils.recuperar_dados_ERA5(dataset_, product_type, variable, ano)


    # Somente para processamento LOCAL, deve ser reescrita
    file_name    = r"{DATA_PATH_ROOT}ERA5-temperaturas\arquivos_NC\ERA5_{variable}_{ano}.nc".format(ano = ano, DATA_PATH_ROOT = DATA_PATH_ROOT, variable = variable)
    os.rename(ret_download, file_name)

    # Converte os dados de netcdf4 para um Dataframe Spark  -> Bronze
    df_spark     = utils.converter_netcdf4_Spark_DF(spark, file_name)


    df_temperatura_final = transform_data(df_spark)


    # Escreve os dados em formato csv
    csv_file_name = f"ERA5_t2m_{ano}.csv"
    csv_path = \
        r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_CSV".format(DATA_PATH_ROOT = DATA_PATH_ROOT)


    utils.write_data_csv(df_temperatura_final, csv_path, csv_file_name)



    finish = datetime(2026, 8, 16).now()
    print(f" - Concluído: {csv_file_name} - {finish} - {(finish - start)} \n")




In [ ]:
# df_temperatura_final.printSchema()
# df_temperatura_final.show(10, False)